# Laboratório — Regularização L2, L1 e early stopping

Construiremos uma MLP de uma camada oculta em **NumPy puro** e auditaremos regularização sem autograd. O conjunto de treino é deliberadamente pequeno e ruidoso para tornar capacidade e sobreajuste observáveis.

**Dependências mínimas:** Python 3.11, NumPy 1.26 e Matplotlib 3.8.  
**Seed canônica:** `20260920`.  
**Dados:** geração sintética documentada; nenhuma fonte externa ou dado pessoal.

O teste fica lacrado durante todas as decisões e é consultado uma única vez após a seleção global.

## 1. Ambiente

Usaremos `float64` para tornar os gradient checks sensíveis. A implementação não depende de scikit-learn, PyTorch, TensorFlow, JAX ou motores de diferenciação automática.

In [ ]:
import platform
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

SEED = 20260920
WEIGHT_KEYS = ("W1", "W2")
np.set_printoptions(precision=6, suppress=True)

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
print(f"Seed: {SEED}")

assert tuple(map(int, np.__version__.split('.')[:2])) >= (1, 26)
assert tuple(map(int, matplotlib.__version__.split('.')[:2])) >= (3, 8)

## 2. Dados e protocolo

A função geradora é $y=\sin(3x)+0{,}25x+\epsilon$. O treino contém apenas 20 pontos com ruído padrão 0,40; validação e teste são gerados por RNGs independentes, com 200 pontos e ruído 0,12.

Não há transformação aprendida nos três conjuntos. Cada observação é a unidade de análise.

**Descrição do gráfico:** pontos laranja de treino aparecem mais dispersos ao redor da curva verdadeira preta; validação azul cobre o mesmo domínio. O teste não é desenhado para permanecer lacrado.

In [ ]:
def true_function(x):
    return np.sin(3 * x) + 0.25 * x


rng_train = np.random.default_rng(SEED)
rng_val = np.random.default_rng(SEED + 1)
rng_test = np.random.default_rng(SEED + 2)

X_train = np.sort(rng_train.uniform(-2, 2, 20))[:, None]
y_train = true_function(X_train[:, 0]) + rng_train.normal(0, 0.40, len(X_train))
X_val = np.linspace(-2, 2, 200)[:, None]
y_val = true_function(X_val[:, 0]) + rng_val.normal(0, 0.12, len(X_val))
X_test = np.linspace(-1.99, 1.99, 200)[:, None]
y_test = true_function(X_test[:, 0]) + rng_test.normal(0, 0.12, len(X_test))

assert X_train.shape == (20, 1)
assert X_val.shape == X_test.shape == (200, 1)
assert not np.array_equal(y_val, y_test)

grid = np.linspace(-2, 2, 400)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(grid, true_function(grid), color="black", label="função sem ruído")
ax.scatter(X_train[:, 0], y_train, color="tab:orange", label="treino", zorder=3)
ax.scatter(X_val[:, 0], y_val, s=10, alpha=0.35, color="tab:blue", label="validação")
ax.set(xlabel="x", ylabel="y", title="Treino pequeno e validação independente")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## 3. MLP e contagem de parâmetros

A arquitetura $1\rightarrow128\rightarrow1$ usa `tanh`. São $1\times128+128+128\times1+1=385$ parâmetros para apenas 20 exemplos de treino. Isso indica alta capacidade, mas não determina sozinho a generalização.

In [ ]:
def init_mlp(width=128, seed=7):
    rng = np.random.default_rng(seed)
    return {
        "W1": rng.normal(0, 1.0, size=(1, width)),
        "b1": np.zeros(width),
        "W2": rng.normal(0, np.sqrt(1 / width), size=(width, 1)),
        "b2": np.zeros(1),
    }


def clone_params(params):
    return {name: value.copy() for name, value in params.items()}


def parameter_count(params):
    return sum(value.size for value in params.values())


def forward(params, X):
    Z1 = X @ params["W1"] + params["b1"]
    H1 = np.tanh(Z1)
    predictions = (H1 @ params["W2"] + params["b2"])[:, 0]
    return predictions, (X, Z1, H1)


base_params = init_mlp()
print("Parâmetros:", parameter_count(base_params))
print("Exemplos de treino:", len(X_train))
assert parameter_count(base_params) == 385

## 4. Loss de dados e backward manual

Adotamos $J_{dados}=\frac{1}{2N}\sum_i(\hat y_i-y_i)^2$. O fator $1/2$ simplifica a derivada. Todas as reduções são médias.

In [ ]:
def data_loss(params, X, y):
    predictions, _ = forward(params, X)
    return float(0.5 * np.mean((predictions - y) ** 2))


def data_gradients(params, X, y):
    predictions, (X_cached, _, H1) = forward(params, X)
    batch_size = len(X)
    d_predictions = (predictions - y)[:, None] / batch_size
    dH1 = d_predictions @ params["W2"].T
    dZ1 = dH1 * (1 - H1**2)
    return {
        "W1": X_cached.T @ dZ1,
        "b1": dZ1.sum(axis=0),
        "W2": H1.T @ d_predictions,
        "b2": d_predictions.sum(axis=0),
    }


grads = data_gradients(base_params, X_train, y_train)
assert set(grads) == set(base_params)
assert all(grads[name].shape == base_params[name].shape for name in base_params)
assert all(np.isfinite(grad).all() for grad in grads.values())
print(f"Loss inicial de treino: {data_loss(base_params, X_train, y_train):.6f}")
print("Shapes do backward: aprovados")

## 5. Gradient check da loss de dados

Verificaremos coordenadas representativas de pesos e biases por diferenças centrais. O denominador do erro relativo evita que valores próximos de zero dominem a razão.

In [ ]:
def central_difference(params, name, index, objective, h=1e-5):
    plus = clone_params(params)
    minus = clone_params(params)
    plus[name][index] += h
    minus[name][index] -= h
    return (objective(plus) - objective(minus)) / (2 * h)


coordinates = [("W1", (0, 3)), ("b1", (7,)), ("W2", (11, 0)), ("b2", (0,))]
relative_errors = []
for name, index in coordinates:
    numerical = central_difference(
        base_params, name, index, lambda p: data_loss(p, X_train, y_train)
    )
    analytical = grads[name][index]
    error = abs(numerical - analytical) / max(1e-12, abs(numerical) + abs(analytical))
    relative_errors.append(error)
    print(name, index, f"analítico={analytical:.9f}", f"numérico={numerical:.9f}", f"erro={error:.3e}")

max_data_gradient_error = float(max(relative_errors))
assert max_data_gradient_error < 1e-8

## 6. L2: objetivo, gradiente e política de bias

Somente `W1` e `W2` entram em $\frac\lambda2\sum_l\|W_l\|_F^2$. O teste numérico cobre pesos e biases: biases devem continuar iguais ao gradiente de dados.

In [ ]:
def l2_penalty(params, strength):
    return 0.5 * strength * sum(np.sum(params[name] ** 2) for name in WEIGHT_KEYS)


def total_l2_loss(params, X, y, strength):
    return data_loss(params, X, y) + l2_penalty(params, strength)


def add_l2_gradients(data_grads, params, strength):
    result = {name: grad.copy() for name, grad in data_grads.items()}
    for name in WEIGHT_KEYS:
        result[name] += strength * params[name]
    return result


strength_check = 0.03
l2_grads = add_l2_gradients(grads, base_params, strength_check)
l2_errors = []
for name, index in coordinates:
    numerical = central_difference(
        base_params,
        name,
        index,
        lambda p: total_l2_loss(p, X_train, y_train, strength_check),
    )
    analytical = l2_grads[name][index]
    error = abs(numerical - analytical) / max(1e-12, abs(numerical) + abs(analytical))
    l2_errors.append(error)

max_l2_gradient_error = float(max(l2_errors))
print(f"Erro relativo máximo com L2: {max_l2_gradient_error:.3e}")
print("Bias b1 inalterado pela penalidade:", np.array_equal(l2_grads["b1"], grads["b1"]))
assert max_l2_gradient_error < 1e-8
assert np.array_equal(l2_grads["b1"], grads["b1"])
assert np.array_equal(l2_grads["b2"], grads["b2"])
assert not np.array_equal(l2_grads["W1"], grads["W1"])

## 7. Equivalência L2–weight decay no SGD puro

Sob a convenção desta aula, `W - eta*(g + lambda*W)` deve coincidir com `(1 - eta*lambda)*W - eta*g`.

In [ ]:
W = np.array([[2.0, -1.0], [0.5, 3.0]])
g = np.array([[0.4, 0.2], [-0.1, 0.7]])
eta, strength = 0.05, 0.1
coupled = W - eta * (g + strength * W)
decayed = (1 - eta * strength) * W - eta * g
decay_error = float(np.max(np.abs(coupled - decayed)))

print(f"Erro máximo entre as formas: {decay_error:.3e}")
assert decay_error < 1e-15

## 8. L1 proximal e a quina em zero

O operador $S_\tau(z)=\operatorname{sign}(z)\max(|z|-\tau,0)$ produz zeros exatos. A atualização ingênua por `sign` pode atravessar zero sem permanecer nele.

In [ ]:
def soft_threshold(value, threshold):
    if threshold < 0:
        raise ValueError("threshold deve ser não negativo")
    return np.sign(value) * np.maximum(np.abs(value) - threshold, 0.0)


z = np.array([0.10, 0.70, -0.50, 0.0])
prox = soft_threshold(z, 0.20)
print("Entrada:", z)
print("Soft-threshold:", prox)
assert np.allclose(prox, np.array([0.0, 0.5, -0.3, 0.0]))

w = 0.015
naive = w - 0.1 * 0.2 * np.sign(w)
proximal = float(soft_threshold(np.array(w), 0.1 * 0.2))
print(f"Step ingênuo: {naive:.6f} | proximal: {proximal:.6f}")
assert np.isclose(naive, -0.005)
assert proximal == 0.0

## 9. Laço de treinamento comum

A função recebe apenas treino e validação. O teste nem sequer aparece na assinatura. L2 modifica gradientes; L1 usa um step proximal após o gradiente de dados. Early stopping avalia a cada dez steps, usa `min_delta` e restaura o melhor snapshot.

In [ ]:
def weight_norm(params):
    return float(np.sqrt(sum(np.sum(params[name] ** 2) for name in WEIGHT_KEYS)))


def count_exact_weight_zeros(params):
    return int(sum(np.count_nonzero(params[name] == 0) for name in WEIGHT_KEYS))


def train_model(
    initial_params,
    X_train,
    y_train,
    X_val,
    y_val,
    *,
    steps=8000,
    learning_rate=0.05,
    penalty="none",
    strength=0.0,
    early_stopping=False,
    eval_every=10,
    patience_evals=60,
    min_delta=1e-6,
):
    params = clone_params(initial_params)
    best_params = clone_params(params)
    best_val = np.inf
    best_step = 0
    waiting = 0
    history = []
    stop_step = steps

    for step in range(1, steps + 1):
        grads = data_gradients(params, X_train, y_train)
        if penalty == "l2":
            grads = add_l2_gradients(grads, params, strength)

        for name in params:
            params[name] -= learning_rate * grads[name]

        if penalty == "l1":
            for name in WEIGHT_KEYS:
                params[name] = soft_threshold(params[name], learning_rate * strength)

        if step % eval_every == 1:
            train_value = data_loss(params, X_train, y_train)
            val_value = data_loss(params, X_val, y_val)
            history.append((step, train_value, val_value, weight_norm(params)))
            if val_value < best_val - min_delta:
                best_val = val_value
                best_step = step
                best_params = clone_params(params)
                waiting = 0
            else:
                waiting += 1
            if early_stopping and waiting >= patience_evals:
                params = clone_params(best_params)
                stop_step = step
                break

    return {
        "params": params,
        "history": np.asarray(history),
        "best_val": float(best_val),
        "best_step": int(best_step),
        "stop_step": int(stop_step),
    }


smoke = train_model(base_params, X_train, y_train, X_val, y_val, steps=20)
assert smoke["history"].shape[1] == 4
assert all(np.isfinite(value).all() for value in smoke["params"].values())
print("Smoke test do laço: aprovado")

## 10. Busca de L2 somente na validação

Todos os candidatos partem do mesmo snapshot e recebem 8.000 updates full-batch. A tabela separa loss preditiva e norma; o teste continua intocado.

In [ ]:
l2_candidates = [0.0, 1e-4, 1e-3, 1e-2]
l2_runs = {}
for candidate in l2_candidates:
    run = train_model(
        base_params,
        X_train,
        y_train,
        X_val,
        y_val,
        penalty="l2" if candidate > 0 else "none",
        strength=candidate,
    )
    l2_runs[candidate] = run
    params = run["params"]
    print(
        f"lambda={candidate:>6g} | treino={data_loss(params, X_train, y_train):.6f} "
        f"| validação={data_loss(params, X_val, y_val):.6f} | ||W||={weight_norm(params):.6f}"
    )

selected_l2 = min(l2_candidates, key=lambda value: data_loss(l2_runs[value]["params"], X_val, y_val))
print("Lambda L2 selecionado:", selected_l2)
assert selected_l2 == 1e-3
assert weight_norm(l2_runs[1e-2]["params"]) < weight_norm(l2_runs[0.0]["params"])
assert data_loss(l2_runs[1e-2]["params"], X_train, y_train) > data_loss(l2_runs[0.0]["params"], X_train, y_train)

**Descrição do gráfico:** a curva sem regularização reduz a loss de treino, mas a validação atinge um mínimo e piora. L2 com $\lambda=10^{-3}$ mantém validação menor ao fim do orçamento.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for candidate, color in [(0.0, "tab:orange"), (1e-3, "tab:blue")]:
    history = l2_runs[candidate]["history"]
    label = "sem regularização" if candidate == 0 else r"L2, $\lambda=10^{-3}$"
    axes[0].plot(history[:, 0], history[:, 1], color=color, linestyle="--", label=label + " — treino")
    axes[0].plot(history[:, 0], history[:, 2], color=color, label=label + " — validação")
    axes[1].plot(history[:, 0], history[:, 3], color=color, label=label)
axes[0].set(xlabel="step", ylabel="loss de dados", title="Treino e validação")
axes[1].set(xlabel="step", ylabel=r"$||W||_2$", title="Norma dos pesos")
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend()
plt.tight_layout()
plt.show()

## 11. Early stopping e restauração

Usaremos o candidato sem penalidade. A execução deve parar depois do melhor step e devolver exatamente o melhor snapshot, não o estado que acionou patience.

In [ ]:
early_run = train_model(
    base_params,
    X_train,
    y_train,
    X_val,
    y_val,
    penalty="none",
    early_stopping=True,
    patience_evals=60,
    min_delta=1e-6,
)
early_val = data_loss(early_run["params"], X_val, y_val)
unregularized_final_val = data_loss(l2_runs[0.0]["params"], X_val, y_val)

print(f"Melhor step: {early_run['best_step']}")
print(f"Parada: {early_run['stop_step']}")
print(f"Validação restaurada: {early_val:.9f}")
print(f"Validação sem parada, step 8000: {unregularized_final_val:.9f}")

assert early_run["stop_step"] > early_run["best_step"]
assert early_run["stop_step"] < 8000
assert np.isclose(early_val, early_run["best_val"])
assert early_val < unregularized_final_val

## 12. L1 proximal e esparsidade exata

Dois candidatos são avaliados apenas na validação. Contaremos zeros por igualdade exata, sem esconder um limiar de pós-processamento.

In [ ]:
l1_candidates = [1e-4, 1e-3]
l1_runs = {}
for candidate in l1_candidates:
    run = train_model(
        base_params,
        X_train,
        y_train,
        X_val,
        y_val,
        penalty="l1",
        strength=candidate,
    )
    l1_runs[candidate] = run
    params = run["params"]
    print(
        f"lambda={candidate:g} | treino={data_loss(params, X_train, y_train):.6f} "
        f"| validação={data_loss(params, X_val, y_val):.6f} "
        f"| zeros exatos={count_exact_weight_zeros(params)}/256"
    )

selected_l1 = min(l1_candidates, key=lambda value: data_loss(l1_runs[value]["params"], X_val, y_val))
assert selected_l1 == 1e-4
assert count_exact_weight_zeros(l1_runs[1e-4]["params"]) >= 10
assert count_exact_weight_zeros(l1_runs[1e-3]["params"]) >= 90
print("Lambda L1 selecionado:", selected_l1)

## 13. Capacidade como ablação, não veredito

Treinaremos larguras 2, 8, 32 e 128 pelo mesmo orçamento. Mais parâmetros reduzem a loss de treino neste problema, mas a validação não é função monotônica universal da largura.

In [ ]:
capacity_rows = []
for width in [2, 8, 32, 128]:
    initial = init_mlp(width=width, seed=7)
    run = train_model(initial, X_train, y_train, X_val, y_val, steps=4000)
    params = run["params"]
    capacity_rows.append(
        (width, parameter_count(params), data_loss(params, X_train, y_train), data_loss(params, X_val, y_val))
    )
for row in capacity_rows:
    print(f"largura={row[0]:3d} | parâmetros={row[1]:3d} | treino={row[2]:.6f} | validação={row[3]:.6f}")

assert capacity_rows[0][1] == 7
assert capacity_rows[-1][1] == 385
assert capacity_rows[-1][2] < capacity_rows[0][2]

## 14. Seleção global e única consulta ao teste

Compararemos, pela validação, o melhor L2, o melhor L1 e early stopping. Somente a configuração vencedora será aplicada ao teste. O contador torna esse contrato executável.

In [ ]:
candidates = {
    f"L2 lambda={selected_l2:g}": l2_runs[selected_l2]["params"],
    f"L1 lambda={selected_l1:g}": l1_runs[selected_l1]["params"],
    "early stopping": early_run["params"],
}
validation_scores = {name: data_loss(params, X_val, y_val) for name, params in candidates.items()}
selected_name = min(validation_scores, key=validation_scores.get)
selected_params = candidates[selected_name]

test_reads = 0
def evaluate_test_once(params):
    global test_reads
    test_reads += 1
    if test_reads > 1:
        raise RuntimeError("teste consultado mais de uma vez")
    return data_loss(params, X_test, y_test)


selected_test_loss = evaluate_test_once(selected_params)
print("Scores de validação:", validation_scores)
print("Configuração selecionada:", selected_name)
print(f"Loss final no teste reservado: {selected_test_loss:.9f}")

assert selected_name == "L2 lambda=0.001"
assert test_reads == 1
assert selected_test_loss < 0.06

## 15. Auditoria final

Os grupos abaixo consolidam derivadas, convenções, separação de dados, esparsidade, early stopping e teste reservado.

In [ ]:
checks = {
    "splits_independentes": not np.array_equal(y_val, y_test),
    "capacidade_contada": parameter_count(base_params) == 385,
    "backward_shapes": all(grads[name].shape == base_params[name].shape for name in base_params),
    "gradiente_dados": max_data_gradient_error < 1e-8,
    "gradiente_l2": max_l2_gradient_error < 1e-8,
    "bias_excluido": np.array_equal(l2_grads["b2"], grads["b2"]),
    "weight_decay_equivalente": decay_error < 1e-15,
    "proximal_zero": proximal == 0.0,
    "l2_selecionado_na_validacao": selected_l2 == 1e-3,
    "early_stop_restaurado": np.isclose(early_val, early_run["best_val"]),
    "l1_esparso": count_exact_weight_zeros(l1_runs[1e-3]["params"]) >= 90,
    "capacidade_ablacionada": capacity_rows[-1][2] < capacity_rows[0][2],
    "teste_consultado_uma_vez": test_reads == 1,
    "teste_finito": np.isfinite(selected_test_loss),
}
assert all(checks.values())
print(f"Auditoria final: {sum(checks.values())}/{len(checks)} grupos aprovados")

## Conclusões

1. Os gradient checks da loss de dados e do objetivo L2 ficaram abaixo de $10^{-8}$.
2. L2 com $\lambda=10^{-3}$ foi escolhida pela validação e reduziu a norma dos pesos.
3. Early stopping restaurou o melhor snapshot, anterior ao step que encerrou o treino.
4. L1 proximal gerou zeros exatos, enquanto o step ingênuo atravessou a origem.
5. A configuração global foi decidida sem teste; a loss reservada foi calculada uma única vez.

O experimento é sintético e pequeno. Ele valida mecanismos e metodologia, não prova que um regularizador vencerá em qualquer arquitetura ou população.